In [ ]:
from dash import Dash, html, dcc, Input, Output, callback
import dash_ag_grid as dag
import pandas as pd
from utils import *

In [ ]:
def format_data(species):
        return species.replace("'", "").replace("[", "").replace("]", "").replace('"', "").lower()

@callback(
    Output('population-table', 'figure'),
    Input('species-dropdown', 'value')
)
def create_gender_pop_table(selected_species):
    aviary_info = general_df[general_df["Aviary"] == AVIARY]

    native_species = aviary_info["species"].iloc[0].split(", ")

    native_species = [format_data(s) for s in native_species]
    gender_list = aviary_info["individuals_genders (m.f.u)"].iloc[0].split(", ")
    gender_list = [format_data(g).split('.') for g in gender_list]

    males = [g[0] for g in gender_list]
    females = [g[1] for g in gender_list]
    unknowns = [g[2] for g in gender_list]

    plot_df = pd.DataFrame({
		"Species": native_species,
		"Males": males,
		"Females": females,
		"Unknown": unknowns
	})

    plot_df = plot_df[plot_df["Species"].isin(selected_species)]

    table_fig = go.Figure(data=[go.Table(
        header=dict(values=['Species', 'Males', 'Females', 'Unknown'])
        , cells=dict(values=[plot_df['Species'], plot_df['Males'], plot_df['Females'], plot_df['Unknown']]))
    ])

    table_fig.update_layout(title="Population Table", transition_duration=500)

    return table_fig


In [ ]:
AVIARY = "Zoo Eindhoven, Large Aviary"
general_df = pd.read_csv("general_aviary_data.csv")
aviary_df = pd.read_excel("metadata_aviaries/fl_zoo_eindhoven_20250308_meta.xlsx")

# Initialize the app
app = Dash()

# App layout
native_species=[sp for sp in format_data(general_df[general_df['Aviary'] == AVIARY]['species'].iloc[0]).split(",")]

app.layout = html.Div(children=[
    html.Div(children= [
    html.Label('Bird Species'),
    dcc.Dropdown(
        id='species-dropdown',
        options=[sp for sp in native_species],
        value=[sp for sp in native_species],
        multi=True,
    ),

    html.Label('Aviary Population Table'),
    dcc.Graph(id='population-table')
    ])
])

# Run the app
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="external")
